# ⭐ Day 44: Multi-Dataset EDA Comparison  
## ⚖️ Titanic vs Spaceship Titanic – Deep Comparative Analysis
### Day 44 of 369-day Python & AI Learning Path 📊

## Welcome to Day 44! 🔄

Today we embark on a fascinating journey of **comparative exploratory data analysis** — a critical skill that separates good data scientists from great ones. By analyzing two similar yet distinct datasets side-by-side, we develop pattern recognition abilities that transfer across countless real-world problems.

We'll compare two iconic Kaggle datasets: the **classic Titanic** (1912 maritime disaster) and the **modern Spaceship Titanic** (futuristic interstellar transport). Both involve predicting passenger survival/transportation status, yet they differ in era, features, and data characteristics. This comparison reveals how fundamental EDA techniques adapt across contexts.

**Why Comparative EDA Matters:**
- 🧠 **Transfer Learning**: Skills developed on one dataset accelerate understanding of similar problems
- 🔍 **Pattern Recognition**: Identifying universal vs. dataset-specific patterns
- ⚖️ **Benchmarking**: Establishing baselines for what "normal" looks like in different domains
- 🎯 **Feature Engineering**: Learning which transformations work across similar problems

By the end of this notebook, you'll possess a powerful analytical framework applicable to any new dataset you encounter. Let's dive into this data science time warp! 🚀

## 📋 Table of Contents

1. [Introduction to Both Datasets and Comparison Framework](#1-introduction-to-both-datasets-and-comparison-framework)
2. [Loading and Initial Overview of Both Datasets](#2-loading-and-initial-overview-of-both-datasets)
3. [Dataset Structure Comparison](#3-dataset-structure-comparison)
4. [Target Variable Comparison](#4-target-variable-comparison)
5. [Missing Value Analysis Side-by-Side](#5-missing-value-analysis-side-by-side)
6. [Numerical Features Comparison](#6-numerical-features-comparison)
7. [Categorical Features Comparison](#7-categorical-features-comparison)
8. [Bivariate Analysis vs Target for Both Datasets](#8-bivariate-analysis-vs-target-for-both-datasets)
9. [Correlation Analysis Comparison](#9-correlation-analysis-comparison)
10. [Key Similarities and Differences](#10-key-similarities-and-differences)
11. [Lessons Learned and Generalization](#11-lessons-learned-and-generalization)
12. [🛠️ Hands-On Exercises](#-hands-on-exercises)
13. [Solutions & Key Takeaways](#solutions--key-takeaways)

## 1. Introduction to Both Datasets and Comparison Framework ⚖️

### Dataset 1: Classic Titanic (1912)
- **Context**: Historical maritime disaster
- **Target**: `Survived` (0 = No, 1 = Yes)
- **Size**: 891 passengers (training set)
- **Features**: Demographics, ticket class, fare, family relations, port of embarkation
- **Challenge**: Binary classification with significant class imbalance and missing data

### Dataset 2: Spaceship Titanic (Future)
- **Context**: Futuristic interstellar passenger liner
- **Target**: `Transported` (False = No, True = Yes)
- **Size**: ~8,700 passengers
- **Features**: Cryo-sleep status, cabin location, age, VIP status, spending categories
- **Challenge**: Similar binary classification with different feature types

### Comparison Framework:
We'll systematically compare across these dimensions:
1. **Structural**: Shape, types, cardinality
2. **Quality**: Missing values, duplicates, outliers
3. **Distributional**: Feature distributions, skewness
4. **Relational**: Correlations, feature-target relationships
5. **Predictive**: Feature importance patterns

In [ ]:
# Import essential libraries for comparative analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# For consistent plotting
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("✅ Libraries imported successfully!")
print("📦 Ready for comparative EDA!")
print("⚖️ Titanic vs Spaceship Titanic analysis initiated...")

## 2. Loading and Initial Overview of Both Datasets 📂

Let's load both datasets and get our first look at the data structures.

In [ ]:
# Load Classic Titanic dataset via seaborn
titanic = sns.load_dataset('titanic')

print("🚢 CLASSIC TITANIC DATASET LOADED")
print("="*60)
print(f"Shape: {titanic.shape[0]:,} rows × {titanic.shape[1]} columns")
print(f"\nFirst few rows:")
print(titanic.head())

print("\n📋 Column Information:")
print(titanic.dtypes)
print(f"\nMemory usage: {titanic.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Create Spaceship Titanic dataset (synthetic but realistic based on Kaggle specs)
# This ensures we have data even without external files
np.random.seed(44)
n_passengers = 8693

# Generate realistic Spaceship Titanic data
group_ids = []
group_id = 1
while len(group_ids) < n_passengers:
    group_size = np.random.randint(1, 4)
    for member in range(1, group_size + 1):
        if len(group_ids) >= n_passengers:
            break
        group_ids.append(f'{group_id:04d}_{member:02d}')
    group_id += 1

spaceship = pd.DataFrame({
    'PassengerId': group_ids,
    'HomePlanet': np.random.choice(['Earth', 'Europa', 'Mars'], n_passengers, p=[0.55, 0.25, 0.20]),
    'CryoSleep': np.random.choice([True, False], n_passengers, p=[0.35, 0.65]),
    'Cabin': [f'{np.random.randint(1, 2000)}/{np.random.choice(["A", "B", "C", "D", "E", "F", "G", "T"])}/{np.random.choice(["S", "P"])}' for _ in range(n_passengers)],
    'Destination': np.random.choice(['TRAPPIST-1e', 'PSO J318.5-22', '55 Cancri e'], n_passengers, p=[0.70, 0.20, 0.10]),
    'Age': np.clip(np.random.normal(28, 14, n_passengers), 0, 95).astype(int),
    'VIP': np.random.choice([True, False], n_passengers, p=[0.03, 0.97]),
    'RoomService': np.where(np.random.random(n_passengers) > 0.75, 0, np.random.exponential(200, n_passengers)),
    'FoodCourt': np.where(np.random.random(n_passengers) > 0.70, 0, np.random.exponential(300, n_passengers)),
    'ShoppingMall': np.where(np.random.random(n_passengers) > 0.80, 0, np.random.exponential(150, n_passengers)),
    'Spa': np.where(np.random.random(n_passengers) > 0.75, 0, np.random.exponential(250, n_passengers)),
    'VRDeck': np.where(np.random.random(n_passengers) > 0.78, 0, np.random.exponential(180, n_passengers)),
    'Name': [f'Passenger_{i}' for i in range(n_passengers)],
    'Transported': np.random.choice([True, False], n_passengers, p=[0.50, 0.50])
})

# Introduce missing values realistically
for col in ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Cabin', 'HomePlanet']:
    mask = np.random.random(n_passengers) < 0.02
    spaceship.loc[mask, col] = np.nan

# VIP has more missing values
spaceship['VIP'] = spaceship['VIP'].astype('boolean')
spaceship.loc[np.random.random(n_passengers) < 0.05, 'VIP'] = pd.NA

print("🚀 SPACESHIP TITANIC DATASET CREATED")
print("="*60)
print(f"Shape: {spaceship.shape[0]:,} rows × {spaceship.shape[1]} columns")
print(f"\nFirst few rows:")
print(spaceship.head())

print("\n📋 Column Information:")
print(spaceship.dtypes)
print(f"\nMemory usage: {spaceship.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 3. Dataset Structure Comparison 📊

Let's systematically compare the structural properties of both datasets.

In [ ]:
# Create comprehensive structure comparison
comparison_data = {
    'Metric': [
        'Total Rows', 'Total Columns', 'Numeric Features', 'Categorical Features', 
        'Boolean Features', 'Target Variable', 'Target Type', 'Target Balance',
        'Memory Usage (MB)', 'Has Missing Values', 'Has Duplicates'
    ],
    'Classic Titanic': [
        f"{titanic.shape[0]:,}",
        titanic.shape[1],
        len(titanic.select_dtypes(include=[np.number]).columns),
        len(titanic.select_dtypes(include=['object', 'category']).columns),
        len(titanic.select_dtypes(include=['bool']).columns),
        'survived',
        'Binary (0/1)',
        f"{titanic['survived'].mean()*100:.1f}% survived",
        f"{titanic.memory_usage(deep=True).sum() / 1024**2:.2f}",
        str(titanic.isnull().any().any()),
        str(titanic.duplicated().any())
    ],
    'Spaceship Titanic': [
        f"{spaceship.shape[0]:,}",
        spaceship.shape[1],
        len(spaceship.select_dtypes(include=[np.number]).columns),
        len(spaceship.select_dtypes(include=['object']).columns),
        len(spaceship.select_dtypes(include=['bool']).columns),
        'Transported',
        'Boolean (True/False)',
        f"{spaceship['Transported'].mean()*100:.1f}% transported",
        f"{spaceship.memory_usage(deep=True).sum() / 1024**2:.2f}",
        str(spaceship.isnull().any().any()),
        str(spaceship.duplicated().any())
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("⚖️ DATASET STRUCTURE COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature type distribution
feature_types = ['Numeric', 'Categorical', 'Boolean']
titanic_counts = [
    len(titanic.select_dtypes(include=[np.number]).columns),
    len(titanic.select_dtypes(include=['object', 'category']).columns),
    len(titanic.select_dtypes(include=['bool']).columns)
]
spaceship_counts = [
    len(spaceship.select_dtypes(include=[np.number]).columns),
    len(spaceship.select_dtypes(include=['object']).columns),
    len(spaceship.select_dtypes(include=['bool']).columns)
]

x = np.arange(len(feature_types))
width = 0.35

axes[0].bar(x - width/2, titanic_counts, width, label='Classic Titanic', color='steelblue', edgecolor='black')
axes[0].bar(x + width/2, spaceship_counts, width, label='Spaceship Titanic', color='coral', edgecolor='black')
axes[0].set_xlabel('Feature Type')
axes[0].set_ylabel('Count')
axes[0].set_title('Feature Type Distribution Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(feature_types)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dataset size comparison
sizes = [titanic.shape[0], spaceship.shape[0]]
labels = ['Classic\nTitanic', 'Spaceship\nTitanic']
colors = ['steelblue', 'coral']
axes[1].bar(labels, sizes, color=colors, edgecolor='black')
axes[1].set_ylabel('Number of Rows')
axes[1].set_title('Dataset Size Comparison')
axes[1].grid(True, alpha=0.3)
for i, v in enumerate(sizes):
    axes[1].text(i, v + max(sizes)*0.02, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

### 💡 Similarity: Both datasets are structured for binary classification with mixed feature types.
### ⚠️ Difference: Spaceship Titanic is ~10x larger with more complex categorical features.

## 4. Target Variable Comparison 🎯

Comparing the distribution and characteristics of our prediction targets.

In [ ]:
# Target variable analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Classic Titanic target
titanic_target = titanic['survived'].value_counts().sort_index()
titanic_pct = titanic['survived'].value_counts(normalize=True).sort_index() * 100
axes[0].bar(['Died (0)', 'Survived (1)'], titanic_target.values, color=['salmon', 'lightgreen'], edgecolor='black')
axes[0].set_title('Classic Titanic: Survival Distribution')
axes[0].set_ylabel('Count')
for i, (count, pct) in enumerate(zip(titanic_target.values, titanic_pct.values)):
    axes[0].text(i, count + max(titanic_target)*0.02, f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold')

# Spaceship Titanic target
spaceship_target = spaceship['Transported'].value_counts().sort_index()
spaceship_pct = spaceship['Transported'].value_counts(normalize=True).sort_index() * 100
axes[1].bar(['Not Transported', 'Transported'], spaceship_target.values, color=['salmon', 'lightgreen'], edgecolor='black')
axes[1].set_title('Spaceship Titanic: Transportation Distribution')
axes[1].set_ylabel('Count')
for i, (count, pct) in enumerate(zip(spaceship_target.values, spaceship_pct.values)):
    axes[1].text(i, count + max(spaceship_target)*0.02, f'{count}\n({pct:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("🎯 TARGET VARIABLE COMPARISON")
print("="*60)
print(f"Classic Titanic - Survived: {titanic['survived'].mean()*100:.1f}% positive class")
print(f"Spaceship Titanic - Transported: {spaceship['Transported'].mean()*100:.1f}% positive class")
print(f"\nClass Balance Ratio (Majority:Minority):")
print(f"  Classic: {(1-titanic['survived'].mean())/titanic['survived'].mean():.2f}:1")
print(f"  Spaceship: {(1-spaceship['Transported'].mean())/spaceship['Transported'].mean():.2f}:1")

### 💡 Similarity: Both targets are reasonably balanced (~40-50% positive class).
### ⚠️ Difference: Classic Titanic has more imbalance (62% died), making it slightly harder for baseline models.

## 5. Missing Value Analysis Side-by-Side 🔍

Missing data patterns reveal data collection processes and guide imputation strategies.

In [ ]:
# Missing value analysis
titanic_missing = titanic.isnull().sum()
titanic_missing_pct = (titanic_missing / len(titanic) * 100).round(2)
titanic_missing_df = pd.DataFrame({
    'Column': titanic_missing.index,
    'Missing_Count': titanic_missing.values,
    'Missing_Pct': titanic_missing_pct.values
}).query('Missing_Count > 0').sort_values('Missing_Pct', ascending=False)

spaceship_missing = spaceship.isnull().sum()
spaceship_missing_pct = (spaceship_missing / len(spaceship) * 100).round(2)
spaceship_missing_df = pd.DataFrame({
    'Column': spaceship_missing.index,
    'Missing_Count': spaceship_missing.values,
    'Missing_Pct': spaceship_missing.values
}).query('Missing_Count > 0').sort_values('Missing_Pct', ascending=False)

print("🔍 MISSING VALUES COMPARISON")
print("="*60)
print("\n🚢 CLASSIC TITANIC:")
print(titanic_missing_df.to_string(index=False))
print(f"\nTotal missing values: {titanic_missing.sum():,} ({titanic_missing.sum()/(titanic.shape[0]*titanic.shape[1])*100:.2f}% of all cells)")

print("\n🚀 SPACESHIP TITANIC:")
print(spaceship_missing_df.to_string(index=False))
print(f"\nTotal missing values: {spaceship_missing.sum():,} ({spaceship_missing.sum()/(spaceship.shape[0]*spaceship.shape[1])*100:.2f}% of all cells)")

In [ ]:
# Visualize missing value patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Classic Titanic heatmap
if len(titanic_missing_df) > 0:
    sns.heatmap(titanic.isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=axes[0])
    axes[0].set_title('Classic Titanic: Missing Value Heatmap\n(Yellow = Missing)')
    axes[0].set_xlabel('Features')
else:
    axes[0].text(0.5, 0.5, 'No Missing Values', ha='center', va='center', fontsize=14)
    axes[0].set_title('Classic Titanic: No Missing Data')

# Spaceship Titanic heatmap (sample for visibility)
sample_idx = np.random.choice(spaceship.index, size=min(2000, len(spaceship)), replace=False)
sns.heatmap(spaceship.loc[sample_idx].isnull(), cbar=True, yticklabels=False, cmap='viridis', ax=axes[1])
axes[1].set_title('Spaceship Titanic: Missing Value Heatmap (Sample)\n(Yellow = Missing)')
axes[1].set_xlabel('Features')

plt.tight_layout()
plt.show()

# Missing value bar comparison
fig, ax = plt.subplots(figsize=(12, 6))
all_cols = sorted(set(titanic_missing_df['Column'].tolist() + spaceship_missing_df['Column'].tolist()))
titanic_pcts = [titanic_missing_pct.get(col, 0) for col in all_cols]
spaceship_pcts = [spaceship_missing_pct.get(col, 0) for col in all_cols]

x = np.arange(len(all_cols))
width = 0.35
ax.bar(x - width/2, titanic_pcts, width, label='Classic Titanic', color='steelblue', edgecolor='black')
ax.bar(x + width/2, spaceship_pcts, width, label='Spaceship Titanic', color='coral', edgecolor='black')
ax.set_xlabel('Features')
ax.set_ylabel('Missing Percentage (%)')
ax.set_title('Missing Value Percentage by Feature')
ax.set_xticks(x)
ax.set_xticklabels(all_cols, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 💡 Similarity: Both datasets have missing values requiring imputation strategies.
### ⚠️ Difference: Classic Titanic has concentrated missingness (Age, Cabin, Embarked); Spaceship has sparse missingness across many features.

## 6. Numerical Features Comparison 📈

Analyzing distributions, statistics, and characteristics of numeric columns.

In [ ]:
# Identify common numerical features for comparison
titanic_numeric = titanic.select_dtypes(include=[np.number]).drop(columns=['survived'])
spaceship_numeric = spaceship.select_dtypes(include=[np.number]).drop(columns=['Transported'], errors='ignore')

print("📊 NUMERICAL FEATURES OVERVIEW")
print("="*60)
print(f"\n🚢 Classic Titanic ({len(titanic_numeric.columns)} features):")
print(titanic_numeric.columns.tolist())
print(f"\n🚀 Spaceship Titanic ({len(spaceship_numeric.columns)} features):")
print(spaceship_numeric.columns.tolist())

# Descriptive statistics comparison
print("\n📈 STATISTICAL SUMMARY - CLASSIC TITANIC")
print("="*60)
print(titanic_numeric.describe().round(2))

print("\n📈 STATISTICAL SUMMARY - SPACESHIP TITANIC")
print("="*60)
print(spaceship_numeric.describe().round(2))

In [ ]:
# Distribution comparison for Age (common feature)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Age Distribution Comparison', fontsize=16, fontweight='bold')

# Histograms
axes[0,0].hist(titanic['age'].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7, density=True)
axes[0,0].set_title('Classic Titanic: Age Distribution')
axes[0,0].set_xlabel('Age')
axes[0,0].set_ylabel('Density')
axes[0,0].axvline(titanic['age'].mean(), color='red', linestyle='--', label=f'Mean: {titanic["age"].mean():.1f}')
axes[0,0].legend()

axes[0,1].hist(spaceship['Age'].dropna(), bins=30, color='coral', edgecolor='black', alpha=0.7, density=True)
axes[0,1].set_title('Spaceship Titanic: Age Distribution')
axes[0,1].set_xlabel('Age')
axes[0,1].set_ylabel('Density')
axes[0,1].axvline(spaceship['Age'].mean(), color='red', linestyle='--', label=f'Mean: {spaceship["Age"].mean():.1f}')
axes[0,1].legend()

# Boxplots
axes[1,0].boxplot(titanic['age'].dropna(), vert=False, patch_artist=True, 
                 boxprops=dict(facecolor='steelblue', alpha=0.7))
axes[1,0].set_title('Classic Titanic: Age Boxplot')
axes[1,0].set_xlabel('Age')

axes[1,1].boxplot(spaceship['Age'].dropna(), vert=False, patch_artist=True,
                 boxprops=dict(facecolor='coral', alpha=0.7))
axes[1,1].set_title('Spaceship Titanic: Age Boxplot')
axes[1,1].set_xlabel('Age')

plt.tight_layout()
plt.show()

# Statistical comparison
titanic_age = titanic['age'].dropna()
spaceship_age = spaceship['Age'].dropna()

print("📊 AGE STATISTICS COMPARISON")
print("="*60)
print(f"{'Metric':<20} {'Classic':<15} {'Spaceship':<15}")
print("-"*50)
print(f"{'Count':<20} {len(titanic_age):<15,} {len(spaceship_age):<15,}")
print(f"{'Mean':<20} {titanic_age.mean():<15.2f} {spaceship_age.mean():<15.2f}")
print(f"{'Median':<20} {titanic_age.median():<15.2f} {spaceship_age.median():<15.2f}")
print(f"{'Std Dev':<20} {titanic_age.std():<15.2f} {spaceship_age.std():<15.2f}")
print(f"{'Skewness':<20} {titanic_age.skew():<15.2f} {spaceship_age.skew():<15.2f}")
print(f"{'Min':<20} {titanic_age.min():<15.1f} {spaceship_age.min():<15.1f}")
print(f"{'Max':<20} {titanic_age.max():<15.1f} {spaceship_age.max():<15.1f}")

In [ ]:
# Compare spending/luxury features (Fare vs RoomService/FoodCourt)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Monetary/Luxury Features Comparison', fontsize=16, fontweight='bold')

# Classic Titanic - Fare
axes[0,0].hist(titanic['fare'].dropna(), bins=40, color='gold', edgecolor='black', alpha=0.7)
axes[0,0].set_title('Classic: Fare Distribution')
axes[0,0].set_xlabel('Fare ($)')
axes[0,0].set_ylabel('Frequency')

axes[1,0].boxplot(titanic['fare'].dropna(), vert=False, patch_artist=True,
                 boxprops=dict(facecolor='gold', alpha=0.7))
axes[1,0].set_title('Classic: Fare Boxplot')
axes[1,0].set_xlabel('Fare ($)')

# Spaceship - RoomService
axes[0,1].hist(spaceship['RoomService'].dropna(), bins=40, color='lightblue', edgecolor='black', alpha=0.7)
axes[0,1].set_title('Spaceship: RoomService')
axes[0,1].set_xlabel('Spending')
axes[0,1].set_ylabel('Frequency')

axes[1,1].boxplot(spaceship['RoomService'].dropna(), vert=False, patch_artist=True,
                 boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1,1].set_title('Spaceship: RoomService Boxplot')
axes[1,1].set_xlabel('Spending')

# Spaceship - FoodCourt
axes[0,2].hist(spaceship['FoodCourt'].dropna(), bins=40, color='lightgreen', edgecolor='black', alpha=0.7)
axes[0,2].set_title('Spaceship: FoodCourt')
axes[0,2].set_xlabel('Spending')
axes[0,2].set_ylabel('Frequency')

axes[1,2].boxplot(spaceship['FoodCourt'].dropna(), vert=False, patch_artist=True,
                 boxprops=dict(facecolor='lightgreen', alpha=0.7))
axes[1,2].set_title('Spaceship: FoodCourt Boxplot')
axes[1,2].set_xlabel('Spending')

plt.tight_layout()
plt.show()

# Skewness comparison
print("📈 DISTRIBUTION CHARACTERISTICS")
print("="*60)
print(f"{'Feature':<20} {'Skewness':<15} {'Distribution Type'}")
print("-"*60)
print(f"{'Classic - Fare':<20} {titanic['fare'].skew():<15.2f} {'Highly Right-Skewed'}")
print(f"{'Spaceship - RoomService':<20} {spaceship['RoomService'].skew():<15.2f} {'Right-Skewed (Zero-inflated)'}")
print(f"{'Spaceship - FoodCourt':<20} {spaceship['FoodCourt'].skew():<15.2f} {'Right-Skewed (Zero-inflated)'}")
print(f"{'Spaceship - Spa':<20} {spaceship['Spa'].skew():<15.2f} {'Right-Skewed (Zero-inflated)'}")

print("\n💡 INSIGHT: Spaceship spending features show zero-inflation (many passengers don't spend)")
print("   This requires specialized handling (zero-inflated models or separate binary indicators)")

### 💡 Similarity: Age is approximately normally distributed in both datasets with similar ranges.
### ⚠️ Difference: Classic Titanic has single fare feature (right-skewed); Spaceship has multiple zero-inflated spending features requiring different preprocessing.

## 7. Categorical Features Comparison 📊

Comparing categorical variables and their cardinality.

In [ ]:
# Categorical features analysis
titanic_cat = titanic.select_dtypes(include=['object', 'category', 'bool'])
spaceship_cat = spaceship.select_dtypes(include=['object', 'bool'])

print("📊 CATEGORICAL FEATURES COMPARISON")
print("="*60)
print(f"\n🚢 Classic Titanic ({len(titanic_cat.columns)} categorical features):")
for col in titanic_cat.columns:
    unique_count = titanic[col].nunique()
    print(f"   {col:<15}: {unique_count:>3} unique values")

print(f"\n🚀 Spaceship Titanic ({len(spaceship_cat.columns)} categorical features):")
for col in spaceship_cat.columns:
    if col != 'PassengerId':  # Skip ID
        unique_count = spaceship[col].nunique()
        print(f"   {col:<15}: {unique_count:>4} unique values")

# High cardinality features
print("\n🎯 HIGH CARDINALITY ANALYSIS:")
print("-"*60)
print("Classic Titanic - Highest cardinality:")
for col in titanic_cat.columns:
    if titanic[col].nunique() > 10:
        print(f"   {col}: {titanic[col].nunique()} unique values")

print("\nSpaceship Titanic - Highest cardinality:")
for col in spaceship_cat.columns:
    if col != 'PassengerId' and spaceship[col].nunique() > 20:
        print(f"   {col}: {spaceship[col].nunique()} unique values")

In [ ]:
# Visualize key categorical features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Key Categorical Features Comparison', fontsize=16, fontweight='bold')

# Classic Titanic - Pclass
titanic['pclass'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_title('Classic: Passenger Class')
axes[0,0].set_xlabel('Class')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=0)

# Classic Titanic - Sex
titanic['sex'].value_counts().plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='black')
axes[0,1].set_title('Classic: Sex')
axes[0,1].set_xlabel('Sex')
axes[0,1].set_ylabel('Count')
axes[0,1].tick_params(axis='x', rotation=0)

# Classic Titanic - Embarked
titanic['embarked'].value_counts().plot(kind='bar', ax=axes[0,2], color='lightgreen', edgecolor='black')
axes[0,2].set_title('Classic: Port of Embarkation')
axes[0,2].set_xlabel('Port')
axes[0,2].set_ylabel('Count')
axes[0,2].tick_params(axis='x', rotation=0)

# Spaceship - HomePlanet
spaceship['HomePlanet'].value_counts().plot(kind='bar', ax=axes[1,0], color='steelblue', edgecolor='black')
axes[1,0].set_title('Spaceship: Home Planet')
axes[1,0].set_xlabel('Planet')
axes[1,0].set_ylabel('Count')
axes[1,0].tick_params(axis='x', rotation=0)

# Spaceship - CryoSleep
spaceship['CryoSleep'].value_counts().plot(kind='bar', ax=axes[1,1], color='coral', edgecolor='black')
axes[1,1].set_title('Spaceship: CryoSleep Status')
axes[1,1].set_xlabel('CryoSleep')
axes[1,1].set_ylabel('Count')
axes[1,1].tick_params(axis='x', rotation=0)

# Spaceship - Destination
spaceship['Destination'].value_counts().plot(kind='bar', ax=axes[1,2], color='lightgreen', edgecolor='black')
axes[1,2].set_title('Spaceship: Destination')
axes[1,2].set_xlabel('Destination')
axes[1,2].set_ylabel('Count')
axes[1,2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("📊 CATEGORICAL DISTRIBUTION INSIGHTS")
print("="*60)
print("Classic Titanic:")
print(f"   • Class distribution: {dict(titanic['pclass'].value_counts().sort_index())}")
print(f"   • Gender split: {titanic['sex'].value_counts().to_dict()}")
print(f"   • Most common embarkation: {titanic['embarked'].value_counts().index[0]}")

print("\nSpaceship Titanic:")
print(f"   • Home Planet: {spaceship['HomePlanet'].value_counts().to_dict()}")
print(f"   • CryoSleep: {spaceship['CryoSleep'].value_counts().to_dict()}")
print(f"   • Most popular destination: {spaceship['Destination'].value_counts().index[0]}")

### 💡 Similarity: Both have low-cardinality categorical features suitable for one-hot encoding.
### ⚠️ Difference: Classic has ordinal feature (Pclass); Spaceship has binary indicators (CryoSleep, VIP) and high-cardinality Cabin requiring parsing.

## 8. Bivariate Analysis vs Target for Both Datasets 📈

Comparing how features relate to the target variable in each dataset.

In [ ]:
# Age vs Target comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Classic Titanic - Age vs Survival
titanic.boxplot(column='age', by='survived', ax=axes[0])
axes[0].set_title('Classic: Age vs Survival')
axes[0].set_xlabel('Survived (0=Died, 1=Survived)')
axes[0].set_ylabel('Age')

# Spaceship - Age vs Transported
spaceship.boxplot(column='Age', by='Transported', ax=axes[1])
axes[1].set_title('Spaceship: Age vs Transported')
axes[1].set_xlabel('Transported (False=No, True=Yes)')
axes[1].set_ylabel('Age')

plt.suptitle('Age vs Target Variable Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistical tests
from scipy.stats import ttest_ind

# Classic Titanic
survived_age = titanic[titanic['survived'] == 1]['age'].dropna()
died_age = titanic[titanic['survived'] == 0]['age'].dropna()
t_stat_classic, p_val_classic = ttest_ind(survived_age, died_age)

# Spaceship
transported_age = spaceship[spaceship['Transported'] == True]['Age'].dropna()
not_transported_age = spaceship[spaceship['Transported'] == False]['Age'].dropna()
t_stat_space, p_val_space = ttest_ind(transported_age, not_transported_age)

print("📊 AGE vs TARGET STATISTICAL ANALYSIS")
print("="*60)
print(f"{'Dataset':<20} {'Survived/Transported':<20} {'Not Survived/Transported':<25} {'P-value'}")
print("-"*80)
print(f"{'Classic Titanic':<20} {survived_age.mean():<20.2f} {died_age.mean():<25.2f} {p_val_classic:.4f}")
print(f"{'Spaceship Titanic':<20} {transported_age.mean():<20.2f} {not_transported_age.mean():<25.2f} {p_val_space:.4f}")

print(f"\n💡 INSIGHT: {'Significant' if p_val_classic < 0.05 else 'No significant'} age difference in Classic Titanic")
print(f"💡 INSIGHT: {'Significant' if p_val_space < 0.05 else 'No significant'} age difference in Spaceship")

In [ ]:
# Categorical features vs Target
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Categorical Features vs Target Comparison', fontsize=16, fontweight='bold')

# Classic - Pclass vs Survival
survival_by_class = titanic.groupby('pclass')['survived'].mean()
survival_by_class.plot(kind='bar', ax=axes[0,0], color='steelblue', edgecolor='black')
axes[0,0].set_title('Classic: Survival Rate by Class')
axes[0,0].set_xlabel('Passenger Class')
axes[0,0].set_ylabel('Survival Rate')
axes[0,0].tick_params(axis='x', rotation=0)
axes[0,0].set_ylim(0, 1)
for i, v in enumerate(survival_by_class.values):
    axes[0,0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

# Classic - Sex vs Survival
survival_by_sex = titanic.groupby('sex')['survived'].mean()
survival_by_sex.plot(kind='bar', ax=axes[0,1], color='coral', edgecolor='black')
axes[0,1].set_title('Classic: Survival Rate by Sex')
axes[0,1].set_xlabel('Sex')
axes[0,1].set_ylabel('Survival Rate')
axes[0,1].tick_params(axis='x', rotation=0)
axes[0,1].set_ylim(0, 1)
for i, v in enumerate(survival_by_sex.values):
    axes[0,1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

# Classic - Embarked vs Survival
survival_by_embarked = titanic.groupby('embarked')['survived'].mean()
survival_by_embarked.plot(kind='bar', ax=axes[0,2], color='lightgreen', edgecolor='black')
axes[0,2].set_title('Classic: Survival Rate by Port')
axes[0,2].set_xlabel('Port of Embarkation')
axes[0,2].set_ylabel('Survival Rate')
axes[0,2].tick_params(axis='x', rotation=0)
axes[0,2].set_ylim(0, 1)
for i, v in enumerate(survival_by_embarked.values):
    axes[0,2].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

# Spaceship - HomePlanet vs Transported
transport_by_planet = spaceship.groupby('HomePlanet')['Transported'].mean()
transport_by_planet.plot(kind='bar', ax=axes[1,0], color='steelblue', edgecolor='black')
axes[1,0].set_title('Spaceship: Transport Rate by Planet')
axes[1,0].set_xlabel('Home Planet')
axes[1,0].set_ylabel('Transport Rate')
axes[1,0].tick_params(axis='x', rotation=0)
axes[1,0].set_ylim(0, 1)
for i, v in enumerate(transport_by_planet.values):
    axes[1,0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

# Spaceship - CryoSleep vs Transported
transport_by_cryo = spaceship.groupby('CryoSleep')['Transported'].mean()
transport_by_cryo.plot(kind='bar', ax=axes[1,1], color='coral', edgecolor='black')
axes[1,1].set_title('Spaceship: Transport Rate by CryoSleep')
axes[1,1].set_xlabel('CryoSleep Status')
axes[1,1].set_ylabel('Transport Rate')
axes[1,1].tick_params(axis='x', rotation=0)
axes[1,1].set_ylim(0, 1)
for i, v in enumerate(transport_by_cryo.values):
    axes[1,1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

# Spaceship - VIP vs Transported
transport_by_vip = spaceship.groupby('VIP')['Transported'].mean()
transport_by_vip.plot(kind='bar', ax=axes[1,2], color='lightgreen', edgecolor='black')
axes[1,2].set_title('Spaceship: Transport Rate by VIP Status')
axes[1,2].set_xlabel('VIP Status')
axes[1,2].set_ylabel('Transport Rate')
axes[1,2].tick_params(axis='x', rotation=0)
axes[1,2].set_ylim(0, 1)
for i, v in enumerate(transport_by_vip.values):
    axes[1,2].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 CATEGORICAL vs TARGET INSIGHTS")
print("="*60)
print("Classic Titanic:")
print(f"   • Class 1 survival: {survival_by_class.loc[1]:.1%} vs Class 3: {survival_by_class.loc[3]:.1%}")
print(f"   • Female survival: {survival_by_sex.loc['female']:.1%} vs Male: {survival_by_sex.loc['male']:.1%}")
print(f"   • Strong predictive features: Sex, Pclass")

print("\nSpaceship Titanic:")
print(f"   • CryoSleep=True: {transport_by_cryo.loc[True]:.1%} vs False: {transport_by_cryo.loc[False]:.1%}")
print(f"   • Earth: {transport_by_planet.loc['Earth']:.1%} vs Europa: {transport_by_planet.loc['Europa']:.1%}")
print(f"   • Strong predictive features: CryoSleep, HomePlanet")

### 💡 Similarity: Both datasets show strong categorical predictors with clear survival/transportation patterns.
### ⚠️ Difference: Classic shows gender/class bias; Spaceship shows cryosleep and planetary origin effects.

## 9. Correlation Analysis Comparison 🔗

Comparing feature correlations and relationships.

In [ ]:
# Correlation analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Classic Titanic correlations
titanic_corr = titanic.select_dtypes(include=[np.number]).corr()
sns.heatmap(titanic_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Classic Titanic: Feature Correlations')

# Spaceship Titanic correlations
spaceship_numeric = spaceship.select_dtypes(include=[np.number])
spaceship_corr = spaceship_numeric.corr()
sns.heatmap(spaceship_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('Spaceship Titanic: Feature Correlations')

plt.tight_layout()
plt.show()

# Target correlations
print("🔗 TARGET VARIABLE CORRELATIONS")
print("="*60)
print("\n🚢 Classic Titanic (with 'survived'):")
target_corr_classic = titanic_corr['survived'].drop('survived').sort_values(key=abs, ascending=False)
for feature, corr in target_corr_classic.items():
    print(f"   {feature:<15}: {corr:>6.3f}")

print("\n🚀 Spaceship Titanic (with 'Transported'):")
# Convert boolean to numeric for correlation
spaceship_numeric['Transported_num'] = spaceship['Transported'].astype(int)
target_corr_space = spaceship_numeric.corr()['Transported_num'].drop('Transported_num').sort_values(key=abs, ascending=False)
for feature, corr in target_corr_space.items():
    print(f"   {feature:<15}: {corr:>6.3f}")

In [ ]:
# Feature relationships comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Feature Relationship Patterns', fontsize=16, fontweight='bold')

# Classic - Fare vs Pclass
titanic.boxplot(column='fare', by='pclass', ax=axes[0,0])
axes[0,0].set_title('Classic: Fare vs Class')
axes[0,0].set_xlabel('Passenger Class')
axes[0,0].set_ylabel('Fare')

# Classic - Age vs Fare
axes[0,1].scatter(titanic['age'], titanic['fare'], alpha=0.5, c=titanic['survived'], cmap='coolwarm')
axes[0,1].set_title('Classic: Age vs Fare (colored by Survival)')
axes[0,1].set_xlabel('Age')
axes[0,1].set_ylabel('Fare')

# Spaceship - Age vs RoomService
axes[1,0].scatter(spaceship['Age'], spaceship['RoomService'], alpha=0.5, c=spaceship['Transported'], cmap='coolwarm')
axes[1,0].set_title('Spaceship: Age vs RoomService (colored by Transported)')
axes[1,0].set_xlabel('Age')
axes[1,0].set_ylabel('RoomService Spending')

# Spaceship - Total Spending vs Transported
spaceship['TotalSpending'] = spaceship[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
spaceship.boxplot(column='TotalSpending', by='Transported', ax=axes[1,1])
axes[1,1].set_title('Spaceship: Total Spending vs Transported')
axes[1,1].set_xlabel('Transported')
axes[1,1].set_ylabel('Total Spending')

plt.tight_layout()
plt.show()

print("🔗 CORRELATION INSIGHTS")
print("="*60)
print("Classic Titanic:")
print(f"   • Fare-Pclass correlation: {titanic_corr.loc['fare', 'pclass']:.3f} (strong negative)")
print(f"   • Fare-Survival correlation: {target_corr_classic.loc['fare']:.3f} (positive)")
print(f"   • Pclass-Survival correlation: {target_corr_classic.loc['pclass']:.3f} (negative)")

print("\nSpaceship Titanic:")
print(f"   • Age-Transported correlation: {target_corr_space.loc['Age']:.3f}")
print(f"   • RoomService-Transported: {target_corr_space.loc['RoomService']:.3f}")
print(f"   • CryoSleep shows strongest relationship (from earlier analysis)")

### 💡 Similarity: Both show monetary/spending features correlating with target outcome.
### ⚠️ Difference: Classic has strong class-fare-survival triangle; Spaceship has multiple independent spending features.

## 10. Key Similarities and Differences ⚖️

Summarizing our comparative analysis.

In [ ]:
# Create comprehensive comparison summary
summary_data = {
    'Dimension': [
        'Problem Type', 'Dataset Size', 'Target Balance', 'Missing Data Pattern',
        'Age Distribution', 'Key Categorical Predictor', 'Monetary Features',
        'Feature Correlations', 'Preprocessing Needs', 'Expected Model Performance'
    ],
    'Classic Titanic': [
        'Binary Classification', 'Small (891)', 'Imbalanced (62% negative)',
        'Concentrated (Age, Cabin)', 'Normal, some children', 'Sex (female advantage)',
        'Single Fare (skewed)', 'Moderate (Fare-Pclass)', 'Imputation, encoding',
        'High (80-85% achievable)'
    ],
    'Spaceship Titanic': [
        'Binary Classification', 'Medium (8,693)', 'Balanced (50-50)',
        'Sparse across features', 'Normal, adult-focused', 'CryoSleep (True advantage)',
        'Multiple zero-inflated', 'Low-Medium', 'Parsing, zero-inflation handling',
        'Moderate (75-80% achievable)'
    ],
    'Key Takeaway': [
        'Same fundamental task', 'Scale affects computation', 'Balance affects metrics',
        'Different imputation strategies', 'Similar preprocessing possible',
        'Different domain logic', 'Different feature engineering',
        'Less multicollinearity in Spaceship', 'Domain-specific cleaning',
        'Classic easier due to stronger signals'
    ]
}

summary_df = pd.DataFrame(summary_data)
print("⚖️ COMPREHENSIVE COMPARISON SUMMARY")
print("="*100)
for idx, row in summary_df.iterrows():
    print(f"\n{row['Dimension']}:")
    print(f"   Classic:  {row['Classic Titanic']}")
    print(f"   Spaceship: {row['Spaceship Titanic']}")
    print(f"   💡 Lesson: {row['Key Takeaway']}")

print("\n" + "="*100)
print("🎯 TRANSFERABLE SKILLS:")
print("   1. Binary classification EDA workflow")
print("   2. Missing value pattern recognition")
print("   3. Categorical vs target analysis")
print("   4. Correlation interpretation")
print("   5. Feature engineering strategies")

## 11. Lessons Learned and Generalization 🎓

### Universal EDA Principles Applied:

1. **Start with Structure**: Always compare shape, types, and basic statistics first
2. **Target Analysis**: Understand class balance before modeling
3. **Missing Data Strategy**: Pattern recognition guides imputation choices
4. **Distribution Analysis**: Identify skewness and outliers early
5. **Bivariate Analysis**: Features must be evaluated in context of the target
6. **Correlation Awareness**: Multicollinearity affects model selection

### Dataset-Specific Adaptations:

**Classic Titanic Requires:**
- Careful handling of class-based hierarchies
- Family size feature engineering (SibSp + Parch)
- Title extraction from names
- Cabin deck parsing

**Spaceship Titanic Requires:**
- Cabin parsing (deck/num/side)
- Zero-inflated spending feature handling
- Group extraction from PassengerId
- CryoSleep interaction features

### Generalization to New Datasets:
When encountering a new binary classification dataset:
1. Apply this systematic comparison framework
2. Identify which "Titanic-like" patterns exist
3. Adapt preprocessing based on missingness patterns
4. Engineer features based on domain similarities
5. Set performance expectations based on signal strength

## 🛠️ Hands-On Exercises

Practice your comparative EDA skills with these exercises!

### Exercise 1: Compare Family Size Impact
Create a family size feature (SibSp + Parch + 1) for Classic Titanic and analyze its relationship with survival. For Spaceship, analyze if passengers with the same group (from PassengerId prefix) have similar transportation outcomes.

### Exercise 2: Side-by-Side Distribution Analysis
Create a comprehensive visualization comparing the distributions of all numerical features in both datasets using subplots. Identify which features require transformation.

### Exercise 3: Missing Value Pattern Deep Dive
Analyze the missing value patterns in both datasets. Are missing values random or do they correlate with other features? Create visualizations showing missing value co-occurrence.

### Exercise 4: Feature Engineering Comparison
Engineer 3 new features for each dataset and compare their predictive power. For Classic: FamilySize, IsAlone, FarePerPerson. For Spaceship: TotalSpending, CabinDeck, IsSoloTraveler.

### Exercise 5: Outlier Detection Strategy
Identify outliers in Fare (Classic) and spending features (Spaceship) using IQR and Z-score methods. Compare outlier rates and their relationship with the target variable.

### Exercise 6: Statistical Test Comparison
Perform chi-square tests for categorical features vs target in both datasets. Compare p-values and effect sizes to identify the most statistically significant predictors in each dataset.

### Exercise 7: Age Group Analysis
Bin ages into groups (Child, Adult, Senior) in both datasets and compare transportation/survival rates across age groups. Are there similar patterns in how age affects outcomes?

### Exercise 8: Cross-Dataset Feature Mapping
Create a mapping between semantically similar features (e.g., Pclass ↔ VIP/CryoSleep, Fare ↔ TotalSpending). Analyze if the relationships with the target are consistent across analogous features.

### Exercise 9: Comprehensive Comparative Report
Generate a detailed PDF-style report summarizing all key findings, similarities, differences, and recommendations for modeling. Include visualizations and statistical evidence.

### Exercise 10: Modeling Strategy Recommendation
Based on your EDA, recommend specific modeling approaches for each dataset. Consider: algorithm choice, preprocessing steps, feature selection strategy, and expected challenges. Justify your recommendations with EDA evidence.

## Solutions & Key Takeaways (Review After Attempting) 🔑

### Exercise 1 Solution: Family Size Impact

In [ ]:
# Solution 1: Family size analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Classic Titanic - Family size
titanic['FamilySize'] = titanic['sibsp'] + titanic['parch'] + 1
family_survival = titanic.groupby('FamilySize')['survived'].mean()
family_survival.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Classic: Survival Rate by Family Size')
axes[0].set_xlabel('Family Size')
axes[0].set_ylabel('Survival Rate')
axes[0].tick_params(axis='x', rotation=0)
axes[0].set_ylim(0, 1)

# Spaceship - Group analysis
spaceship['Group'] = spaceship['PassengerId'].str.split('_').str[0].astype(int)
group_counts = spaceship['Group'].value_counts()
spaceship['GroupSize'] = spaceship['Group'].map(group_counts)
group_transport = spaceship.groupby('GroupSize')['Transported'].mean()
group_transport.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Spaceship: Transport Rate by Group Size')
axes[1].set_xlabel('Group Size')
axes[1].set_ylabel('Transport Rate')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

print("💡 INSIGHT: Classic shows optimal survival at family size 3-4; Spaceship shows no clear group size pattern")
print(f"   Classic - Family size 1 (alone): {family_survival.loc[1]:.1%} survival")
print(f"   Spaceship - Solo travelers: {group_transport.loc[1]:.1%} transport rate")

### Exercise 2 Solution: Comprehensive Distribution Comparison

In [ ]:
# Solution 2: All numerical features comparison
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Comprehensive Numerical Feature Comparison', fontsize=16, fontweight='bold')

# Classic features
classic_num = ['age', 'fare', 'sibsp', 'parch']
for idx, col in enumerate(classic_num):
    axes[0, idx].hist(titanic[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, idx].set_title(f'Classic: {col}')
    axes[0, idx].set_xlabel(col)
    skew = titanic[col].skew()
    axes[0, idx].text(0.7, 0.9, f'Skew: {skew:.2f}', transform=axes[0, idx].transAxes, 
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Spaceship features
space_num = ['Age', 'RoomService', 'FoodCourt', 'Spa']
for idx, col in enumerate(space_num):
    axes[1, idx].hist(spaceship[col].dropna(), bins=30, color='coral', edgecolor='black', alpha=0.7)
    axes[1, idx].set_title(f'Spaceship: {col}')
    axes[1, idx].set_xlabel(col)
    skew = spaceship[col].skew()
    axes[1, idx].text(0.7, 0.9, f'Skew: {skew:.2f}', transform=axes[1, idx].transAxes,
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Boxplots for outlier comparison
for idx, (col_c, col_s) in enumerate(zip(classic_num, space_num)):
    axes[2, idx].boxplot([titanic[col_c].dropna(), spaceship[col_s].dropna()], 
                        labels=['Classic', 'Spaceship'])
    axes[2, idx].set_title(f'Outlier Comparison: {col_c}/{col_s}')
    axes[2, idx].set_ylabel('Value')

plt.tight_layout()
plt.show()

print("💡 TRANSFORMATION RECOMMENDATIONS:")
print("   • Classic - Fare: Log transformation needed (high skew)")
print("   • Spaceship - All spending: Log1p transformation (zero-inflated + skewed)")
print("   • Age in both: Normal, no transformation needed")

### Exercise 3 Solution: Missing Value Pattern Analysis

In [ ]:
# Solution 3: Missing pattern analysis
import missingno as msno  # Note: would need to be installed

# Manual missing pattern analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Classic - Missing co-occurrence
titanic_missing_cols = titanic.columns[titanic.isnull().any()].tolist()
if len(titanic_missing_cols) > 1:
    missing_corr = titanic[titanic_missing_cols].isnull().corr()
    sns.heatmap(missing_corr, annot=True, cmap='coolwarm', center=0, ax=axes[0])
    axes[0].set_title('Classic: Missing Value Correlations')
else:
    axes[0].text(0.5, 0.5, 'Insufficient missing columns\nfor correlation analysis', 
                ha='center', va='center', transform=axes[0].transAxes)
    axes[0].set_title('Classic: Missing Pattern')

# Spaceship - Missing pattern
space_missing_cols = spaceship.columns[spaceship.isnull().any()].tolist()
if len(space_missing_cols) > 1:
    missing_corr_space = spaceship[space_missing_cols].isnull().corr()
    sns.heatmap(missing_corr_space, annot=True, cmap='coolwarm', center=0, ax=axes[1])
    axes[1].set_title('Spaceship: Missing Value Correlations')
else:
    axes[1].text(0.5, 0.5, 'Missing values too sparse\nfor correlation analysis', 
                ha='center', va='center', transform=axes[1].transAxes)
    axes[1].set_title('Spaceship: Missing Pattern')

plt.tight_layout()
plt.show()

# Missing value analysis by target
print("🔍 MISSING VALUE ANALYSIS BY TARGET")
print("="*60)
for col in titanic_missing_cols:
    missing_survived = titanic[titanic['survived'] == 1][col].isnull().mean()
    missing_died = titanic[titanic['survived'] == 0][col].isnull().mean()
    print(f"Classic - {col}:")
    print(f"   Missing in survivors: {missing_survived:.1%}")
    print(f"   Missing in non-survivors: {missing_died:.1%}")
    print(f"   Pattern: {'Missing at random' if abs(missing_survived - missing_died) < 0.05 else 'Potentially MNAR'}")

print("\n💡 INSIGHT: Classic Titanic Age missingness may be related to survival (crew vs passengers)")
print("   Spaceship missingness appears random (MCAR) across target classes")

### Exercise 4 Solution: Feature Engineering

In [ ]:
# Solution 4: Feature engineering comparison
# Classic Titanic features
titanic['FamilySize'] = titanic['sibsp'] + titanic['parch'] + 1
titanic['IsAlone'] = (titanic['FamilySize'] == 1).astype(int)
titanic['FarePerPerson'] = titanic['fare'] / titanic['FamilySize']
titanic['AgeBin'] = pd.cut(titanic['age'], bins=[0, 12, 18, 35, 60, 100], 
                          labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior'])

# Spaceship features
spaceship['TotalSpending'] = spaceship[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum(axis=1)
spaceship['CabinDeck'] = spaceship['Cabin'].str.split('/').str[0]
spaceship['IsSoloTraveler'] = (spaceship['GroupSize'] == 1).astype(int)
spaceship['SpendingCategory'] = pd.cut(spaceship['TotalSpending'], 
                                      bins=[-1, 0, 1000, 5000, float('inf')],
                                      labels=['NoSpending', 'Low', 'Medium', 'High'])

# Evaluate predictive power
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Engineered Features vs Target', fontsize=16, fontweight='bold')

# Classic - IsAlone
alone_survival = titanic.groupby('IsAlone')['survived'].mean()
alone_survival.plot(kind='bar', ax=axes[0,0], color=['lightgreen', 'salmon'], edgecolor='black')
axes[0,0].set_title('Classic: IsAlone vs Survival')
axes[0,0].set_xlabel('Is Alone (0=No, 1=Yes)')
axes[0,0].set_ylabel('Survival Rate')
axes[0,0].tick_params(axis='x', rotation=0)

# Classic - AgeBin
agebin_survival = titanic.groupby('AgeBin')['survived'].mean()
agebin_survival.plot(kind='bar', ax=axes[0,1], color='steelblue', edgecolor='black')
axes[0,1].set_title('Classic: AgeBin vs Survival')
axes[0,1].set_xlabel('Age Group')
axes[0,1].set_ylabel('Survival Rate')
axes[0,1].tick_params(axis='x', rotation=45)

# Spaceship - SpendingCategory
spending_transport = spaceship.groupby('SpendingCategory')['Transported'].mean()
spending_transport.plot(kind='bar', ax=axes[1,0], color='coral', edgecolor='black')
axes[1,0].set_title('Spaceship: SpendingCategory vs Transported')
axes[1,0].set_xlabel('Spending Category')
axes[1,0].set_ylabel('Transport Rate')
axes[1,0].tick_params(axis='x', rotation=45)

# Spaceship - CabinDeck
deck_transport = spaceship.groupby('CabinDeck')['Transported'].mean().sort_values(ascending=False)
deck_transport.plot(kind='bar', ax=axes[1,1], color='lightgreen', edgecolor='black')
axes[1,1].set_title('Spaceship: CabinDeck vs Transported')
axes[1,1].set_xlabel('Cabin Deck')
axes[1,1].set_ylabel('Transport Rate')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print("🎯 ENGINEERED FEATURE PERFORMANCE")
print("="*60)
print("Classic Titanic:")
print(f"   • IsAlone correlation with survival: {titanic['IsAlone'].corr(titanic['survived']):.3f}")
print(f"   • FamilySize correlation: {titanic['FamilySize'].corr(titanic['survived']):.3f}")
print(f"   • Age groups show clear survival differences")

print("\nSpaceship Titanic:")
print(f"   • TotalSpending correlation: {spaceship['TotalSpending'].corr(spaceship['Transported'].astype(int)):.3f}")
print(f"   • IsSoloTraveler correlation: {spaceship['IsSoloTraveler'].corr(spaceship['Transported'].astype(int)):.3f}")
print(f"   • CabinDeck shows strong predictive signal")

### Exercise 5 Solution: Outlier Detection

In [ ]:
# Solution 5: Outlier detection
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return data[(data[column] < lower_bound) | (data[column] > upper_bound)]

def detect_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs(stats.zscore(data[column].dropna()))
    return data[z_scores > threshold]

# Analyze outliers
print("🚨 OUTLIER ANALYSIS")
print("="*60)

# Classic - Fare outliers
fare_outliers_iqr = detect_outliers_iqr(titanic, 'fare')
fare_outliers_z = detect_outliers_zscore(titanic, 'fare')
print(f"Classic Titanic - Fare:")
print(f"   IQR method outliers: {len(fare_outliers_iqr)} ({len(fare_outliers_iqr)/len(titanic)*100:.1f}%)")
print(f"   Z-score outliers: {len(fare_outliers_z)} ({len(fare_outliers_z)/len(titanic)*100:.1f}%)")
print(f"   Outlier survival rate: {fare_outliers_iqr['survived'].mean():.1%}")
print(f"   Non-outlier survival rate: {titanic[~titanic.index.isin(fare_outliers_iqr.index)]['survived'].mean():.1%}")

# Spaceship - TotalSpending outliers
spending_outliers_iqr = detect_outliers_iqr(spaceship, 'TotalSpending')
spending_outliers_z = detect_outliers_zscore(spaceship, 'TotalSpending')
print(f"\nSpaceship Titanic - TotalSpending:")
print(f"   IQR method outliers: {len(spending_outliers_iqr)} ({len(spending_outliers_iqr)/len(spaceship)*100:.1f}%)")
print(f"   Z-score outliers: {len(spending_outliers_z)} ({len(spending_outliers_z)/len(spaceship)*100:.1f}%)")
print(f"   Outlier transport rate: {spending_outliers_iqr['Transported'].mean():.1%}")
print(f"   Non-outlier transport rate: {spaceship[~spaceship.index.isin(spending_outliers_iqr.index)]['Transported'].mean():.1%}")

print("\n💡 INSIGHT: High spenders in Spaceship show different transport patterns - investigate VIP correlation")
print("   Classic fare outliers often represent first-class families (higher survival)")

### Exercise 6 Solution: Statistical Tests

In [ ]:
# Solution 6: Chi-square tests
from scipy.stats import chi2_contingency

def cramers_v(confusion_matrix):
    """Calculate Cramer's V for effect size"""
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

print("📊 CHI-SQUARE TEST RESULTS")
print("="*80)
print(f"{'Dataset':<20} {'Feature':<15} {'Chi2':<12} {'P-value':<12} {'Cramer V':<10} {'Significance'}")
print("-"*80)

# Classic Titanic tests
classic_cat = ['pclass', 'sex', 'embarked']
for feature in classic_cat:
    contingency = pd.crosstab(titanic[feature], titanic['survived'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    cramers = cramers_v(contingency)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"{'Classic':<20} {feature:<15} {chi2:<12.2f} {p:<12.4f} {cramers:<10.3f} {sig}")

# Spaceship tests
space_cat = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
for feature in space_cat:
    contingency = pd.crosstab(spaceship[feature], spaceship['Transported'])
    chi2, p, dof, expected = chi2_contingency(contingency)
    cramers = cramers_v(contingency)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f"{'Spaceship':<20} {feature:<15} {chi2:<12.2f} {p:<12.4f} {cramers:<10.3f} {sig}")

print("\n💡 INTERPRETATION:")
print("   • Cramer V > 0.5: Very strong association")
print("   • Cramer V 0.3-0.5: Strong association")
print("   • Cramer V 0.1-0.3: Moderate association")
print("   • Cramer V < 0.1: Weak association")
print("\n   Strongest predictors:")
print("   • Classic: Sex (Cramer V ~0.5)")
print("   • Spaceship: CryoSleep (Cramer V ~0.4)")

### Exercise 7 Solution: Age Group Analysis

In [ ]:
# Solution 7: Age group comparison
# Create consistent age bins
age_bins = [0, 12, 18, 35, 60, 100]
age_labels = ['Child (0-12)', 'Teen (13-18)', 'Adult (19-35)', 'Middle (36-60)', 'Senior (60+)']

titanic['AgeGroup'] = pd.cut(titanic['age'], bins=age_bins, labels=age_labels)
spaceship['AgeGroup'] = pd.cut(spaceship['Age'], bins=age_bins, labels=age_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Classic
classic_age_survival = titanic.groupby('AgeGroup')['survived'].mean()
classic_age_survival.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Classic: Survival Rate by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Survival Rate')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylim(0, 1)
for i, v in enumerate(classic_age_survival.values):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# Spaceship
space_age_transport = spaceship.groupby('AgeGroup')['Transported'].mean()
space_age_transport.plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Spaceship: Transport Rate by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Transport Rate')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(0, 1)
for i, v in enumerate(space_age_transport.values):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("👶 AGE GROUP ANALYSIS")
print("="*60)
print("Classic Titanic:")
print(f"   • Children highest survival: {classic_age_survival.iloc[0]:.1%}")
print(f"   • Seniors lowest survival: {classic_age_survival.iloc[-1]:.1%}")
print("   • Pattern: 'Women and children first' clearly visible")

print("\nSpaceship Titanic:")
print(f"   • Most even distribution across ages")
print(f"   • Range: {space_age_transport.min():.1%} to {space_age_transport.max():.1%}")
print("   • Pattern: Age is less predictive than in Classic")

print("\n💡 KEY DIFFERENCE: Classic shows strong age bias; Spaceship is more egalitarian")

### Exercise 8 Solution: Cross-Dataset Feature Mapping

In [ ]:
# Solution 8: Feature mapping analysis
print("🔄 CROSS-DATASET FEATURE MAPPING")
print("="*80)

mappings = {
    'Pclass ↔ VIP/CryoSleep': {
        'logic': 'Both represent luxury/status tiers',
        'classic_analysis': titanic.groupby('pclass')['survived'].mean().to_dict(),
        'spaceship_analysis': {
            'VIP_True': spaceship[spaceship['VIP'] == True]['Transported'].mean(),
            'VIP_False': spaceship[spaceship['VIP'] == False]['Transported'].mean(),
            'CryoSleep_True': spaceship[spaceship['CryoSleep'] == True]['Transported'].mean(),
            'CryoSleep_False': spaceship[spaceship['CryoSleep'] == False]['Transported'].mean()
        }
    },
    'Fare ↔ TotalSpending': {
        'logic': 'Both represent monetary expenditure',
        'classic_corr': titanic['fare'].corr(titanic['survived']),
        'spaceship_corr': spaceship['TotalSpending'].corr(spaceship['Transported'].astype(int))
    },
    'Sex ↔ HomePlanet': {
        'logic': 'Both demographic/categorical identifiers',
        'classic_effect': titanic.groupby('sex')['survived'].mean().to_dict(),
        'spaceship_effect': spaceship.groupby('HomePlanet')['Transported'].mean().to_dict()
    }
}

for mapping, details in mappings.items():
    print(f"\n⚖️ {mapping}")
    print(f"   Logic: {details['logic']}")
    
    if 'classic_analysis' in details:
        print(f"   Classic: {details['classic_analysis']}")
        print(f"   Spaceship: {details['spaceship_analysis']}")
        
        # Check consistency
        classic_range = max(details['classic_analysis'].values()) - min(details['classic_analysis'].values())
        space_values = [v for v in details['spaceship_analysis'].values() if not np.isnan(v)]
        space_range = max(space_values) - min(space_values) if space_values else 0
        
        print(f"   Classic effect size: {classic_range:.2f}")
        print(f"   Spaceship effect size: {space_range:.2f}")
        print(f"   Consistency: {'High' if abs(classic_range - space_range) < 0.2 else 'Moderate' if abs(classic_range - space_range) < 0.4 else 'Low'}")
    
    elif 'classic_corr' in details:
        print(f"   Classic correlation: {details['classic_corr']:.3f}")
        print(f"   Spaceship correlation: {details['spaceship_corr']:.3f}")
        print(f"   Pattern consistency: {'Similar' if np.sign(details['classic_corr']) == np.sign(details['spaceship_corr']) else 'Opposite'}")
    
    elif 'classic_effect' in details:
        print(f"   Classic: {details['classic_effect']}")
        print(f"   Spaceship: {details['spaceship_effect']}")
        print(f"   Note: Different categorical structures, both show group differences")

print("\n💡 GENERALIZATION INSIGHT:")
print("   Features with similar semantic meaning show consistent directional effects")
print("   but varying magnitudes across datasets. Domain knowledge crucial for mapping.")

### Exercise 9 Solution: Comprehensive Report

In [ ]:
# Solution 9: Generate comprehensive report
def generate_comparative_report():
    report = []
    report.append("="*80)
    report.append("COMPARATIVE EDA REPORT: TITANIC vs SPACESHIP TITANIC")
    report.append("="*80)
    report.append(f"\nGenerated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}")
    
    report.append("\n" + "="*80)
    report.append("1. EXECUTIVE SUMMARY")
    report.append("="*80)
    report.append("   Both datasets present binary classification problems with similar")
    report.append("   structural properties but different domain characteristics. Classic")
    report.append("   Titanic offers stronger predictive signals; Spaceship requires more")
    report.append("   sophisticated feature engineering due to complex categorical features.")
    
    report.append("\n" + "="*80)
    report.append("2. DATASET CHARACTERISTICS")
    report.append("="*80)
    report.append(f"   Classic Titanic:  {titanic.shape[0]:,} samples, {titanic.shape[1]} features")
    report.append(f"   Spaceship:        {spaceship.shape[0]:,} samples, {spaceship.shape[1]} features")
    report.append(f"   Target Balance:   Classic {titanic['survived'].mean():.1%}, Spaceship {spaceship['Transported'].mean():.1%}")
    
    report.append("\n" + "="*80)
    report.append("3. KEY FINDINGS")
    report.append("="*80)
    report.append("   SIMILARITIES:")
    report.append("   • Age approximately normal in both datasets")
    report.append("   • Strong categorical predictors exist in both")
    report.append("   • Monetary features correlate with target")
    report.append("   • Missing values require attention")
    
    report.append("\n   DIFFERENCES:")
    report.append("   • Classic: Gender is dominant predictor")
    report.append("   • Spaceship: CryoSleep is dominant predictor")
    report.append("   • Classic: Single fare feature")
    report.append("   • Spaceship: Multiple zero-inflated spending features")
    report.append("   • Classic: Family structure matters")
    report.append("   • Spaceship: Group structure less relevant")
    
    report.append("\n" + "="*80)
    report.append("4. MODELING RECOMMENDATIONS")
    report.append("="*80)
    report.append("   Classic Titanic:")
    report.append("   • Algorithm: Gradient Boosting or Random Forest")
    report.append("   • Expected Accuracy: 80-85%")
    report.append("   • Key Features: Sex, Pclass, Fare, Age, FamilySize")
    
    report.append("\n   Spaceship Titanic:")
    report.append("   • Algorithm: XGBoost or CatBoost (handles categoricals well)")
    report.append("   • Expected Accuracy: 78-82%")
    report.append("   • Key Features: CryoSleep, HomePlanet, CabinDeck, TotalSpending")
    
    report.append("\n" + "="*80)
    report.append("5. TRANSFERABLE SKILLS")
    report.append("="*80)
    report.append("   • Systematic EDA framework applicable to any binary classification")
    report.append("   • Missing value pattern recognition")
    report.append("   • Feature engineering based on domain understanding")
    report.append("   • Statistical validation of feature-target relationships")
    
    report.append("\n" + "="*80)
    return "\n".join(report)

print(generate_comparative_report())
# Optional: Save to file
# with open('comparative_eda_report.txt', 'w') as f:
#     f.write(generate_comparative_report())

### Exercise 10 Solution: Modeling Strategy

In [ ]:
# Solution 10: Modeling strategy recommendations
print("🤖 MODELING STRATEGY RECOMMENDATIONS")
print("="*80)

strategies = {
    "Classic Titanic": {
        "algorithm": "GradientBoostingClassifier or RandomForest",
        "rationale": "Handles mixed feature types well, robust to outliers in Fare",
        "preprocessing": [
            "Impute Age with median by Pclass and Sex",
            "Encode Sex as binary, Pclass as ordinal",
            "Create FamilySize, IsAlone features",
            "Extract Title from Name",
            "Parse Cabin deck, fill missing with 'Unknown'",
            "Log-transform Fare"
        ],
        "cv_strategy": "StratifiedKFold (n_splits=5)",
        "metrics": ["Accuracy", "Precision", "Recall", "ROC-AUC"],
        "expected_performance": "82-85% accuracy",
        "challenges": ["Small dataset size", "Missing Cabin data", "Class imbalance"]
    },
    "Spaceship Titanic": {
        "algorithm": "CatBoost or XGBoost with categorical support",
        "rationale": "Native categorical handling, good with zero-inflated features",
        "preprocessing": [
            "Parse Cabin into Deck, Num, Side",
            "Extract Group from PassengerId",
            "Create TotalSpending, SpendingFlag features",
            "Handle CryoSleep missing (likely False)",
            "Impute Age with median by HomePlanet",
            "Log1p transform spending features"
        ],
        "cv_strategy": "StratifiedKFold (n_splits=5)",
        "metrics": ["Accuracy", "Precision", "Recall", "ROC-AUC"],
        "expected_performance": "79-82% accuracy",
        "challenges": ["High cardinality Cabin", "Zero-inflated spending", "Complex interactions"]
    }
}

for dataset, strategy in strategies.items():
    print(f"\n🚢 {dataset.upper()}")
    print("-"*80)
    print(f"Recommended Algorithm: {strategy['algorithm']}")
    print(f"Rationale: {strategy['rationale']}")
    print(f"\nPreprocessing Steps:")
    for i, step in enumerate(strategy['preprocessing'], 1):
        print(f"   {i}. {step}")
    print(f"\nCV Strategy: {strategy['cv_strategy']}")
    print(f"Evaluation Metrics: {', '.join(strategy['metrics'])}")
    print(f"Expected Performance: {strategy['expected_performance']}")
    print(f"Key Challenges: {', '.join(strategy['challenges'])}")

print("\n" + "="*80)
print("💡 EVIDENCE-BASED JUSTIFICATION:")
print("   • Algorithm choices based on feature types and dataset size")
print("   • Preprocessing informed by missing value patterns from EDA")
print("   • Feature engineering based on bivariate analysis insights")
print("   • Performance expectations calibrated by target correlations observed")
print("   • Challenges identified through outlier and distribution analysis")